# Writer Palmyra Vision on Amazon Bedrock Mantle

Writer's Palmyra Vision model on the `bedrock-mantle` endpoint. This is a deliberately instructive case: it is vision-capable but **does not support tool calling**, so it shows both a specialised strength and how to work around a real limitation.

**Models covered in this notebook**

| Model ID | Notes |
|---|---|
| `writer.palmyra-vision-7b` | Vision-language, 7B. Tool calling NOT supported |

### Which API? Chat Completions.
This family is served by the **OpenAI-compatible Chat Completions API** on the
`bedrock-mantle` endpoint, at the bare `/v1` path. The Responses API returns
**400 "does not support this API"** for these models — we prove that in §2 rather
than asking you to take it on trust.

### Self-contained, but see also
Everything you need is here. For deeper background on shared mechanics:
- **Auth (SigV4 + short-term API keys), the three URL paths, model discovery** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Projects, cost attribution, data retention / ZDR, CloudWatch namespace** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas, retry/backoff, service tiers, TTFT measurement** →
  `../00-foundations/03-scaling-tiers-and-latency.ipynb`

### Prerequisites
```bash
pip install openai aws-bedrock-token-generator
```
AWS credentials with `bedrock-mantle:CreateInference` and
`bedrock-mantle:CallWithBearerToken` — both granted by the managed policy
`AmazonBedrockMantleInferenceAccess`.

In [1]:
import json
import sys
import time

sys.path.insert(0, "../_shared")
from mantle import err, parse_json_lenient, post, ttft

REGION = "us-east-1"

VISION = "writer.palmyra-vision-7b"


# Chat-Completions families live at the BARE /v1 path — not /openai/v1
# (that prefix is only for gemma-4, gpt-5.x and grok). See ../00-foundations/01.
PREFIX = "/v1"
BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}"
print("base URL:", BASE_URL)
print("models  :", [VISION])

base URL: https://bedrock-mantle.us-east-1.api.aws/v1
models  : ['writer.palmyra-vision-7b']


## 1. First call

Auth is a short-term Bedrock API key minted from your ambient IAM credentials.
It expires within 12 hours and **cannot be refreshed** — mint a new one instead.
(`../00-foundations/01` shows the self-refreshing provider and the SigV4
alternative that needs no key at all.)

In [2]:
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

# Build the client from a FRESH token — don't construct one at import time and
# reuse it for hours, because the baked-in key expires.
client = OpenAI(api_key=provide_token(region=REGION), base_url=BASE_URL)

completion = client.chat.completions.create(
    model=VISION,
    messages=[{"role": "user", "content": "Explain what a vision-language model does, in two sentences."}],
    max_tokens=250,
)
print(completion.choices[0].message.content)
print("\nusage:", completion.usage.model_dump_json())

 Vision-language models are sophisticated AI systems that seamlessly integrate visual and textual information, enabling them to understand and generate contextually relevant content across both domains. These models can process images and text simultaneously, allowing them to decipher visual cues and use language to producecoherent descriptions or responses, making them powerful tools for tasks that require combining visual and linguistic information.

usage: {"completion_tokens":70,"prompt_tokens":16,"total_tokens":86,"completion_tokens_details":null,"prompt_tokens_details":null}


## 2. Why Chat Completions and not Responses

AWS recommends the Responses API for new applications in general — but
availability is per-model. Probe both surfaces so the 400 is visible:

In [3]:
for api_name, path, body in [
    ("Chat Completions", f"{PREFIX}/chat/completions",
     {"model": VISION, "messages": [{"role": "user", "content": "Reply OK"}],
      "max_tokens": 16}),
    ("Responses (/v1)", f"{PREFIX}/responses",
     {"model": VISION, "input": "Reply OK", "max_output_tokens": 16}),
    ("Responses (/openai/v1)", "/openai/v1/responses",
     {"model": VISION, "input": "Reply OK", "max_output_tokens": 16}),
]:
    code, data = post(path, body, region=REGION)
    print(f"  {api_name:24} -> HTTP {code} {'' if code == 200 else err(data)[:64]}")

  Chat Completions         -> HTTP 200 


  Responses (/v1)          -> HTTP 400 The model 'writer.palmyra-vision-7b' does not support the '/v1/r


  Responses (/openai/v1)   -> HTTP 400 The model 'writer.palmyra-vision-7b' does not support the '/open


Concrete consequences of being Chat-Completions-only:

- **You own the conversation history.** There is no `previous_response_id`
  server-side state on this API — send the full `messages` array each turn.
- **Reasoning content is not returned.** `reasoning_effort` is accepted and the
  model does think, but the OpenAI Chat Completions schema has nowhere to put the
  trace, so you pay for those tokens without seeing them.
- Structured output uses `response_format`, not `text.format`.

## 3. Sampling parameters

This family accepts both `temperature` and `top_p`. That is *not* universal on
mantle — Gemma 4 rejects `top_p`, and Grok rejects `temperature` — so never share
one sampling config across families.

In [4]:
for label, extra in [
    ("temperature=0.7", {"temperature": 0.7}),
    ("temperature=0.0", {"temperature": 0.0}),
    ("top_p=0.95", {"top_p": 0.95}),
    ("both", {"temperature": 0.7, "top_p": 0.95}),
    ("max_tokens=1", {"max_tokens": 1}),
]:
    body = {"model": VISION,
            "messages": [{"role": "user", "content": "Reply OK"}], "max_tokens": 16}
    body.update(extra)
    code, data = post(f"{PREFIX}/chat/completions", body, region=REGION)
    print(f"  {label:18} -> HTTP {code} {'' if code == 200 else err(data)[:60]}")

  temperature=0.7    -> HTTP 200 


  temperature=0.0    -> HTTP 200 


  top_p=0.95         -> HTTP 200 


  both               -> HTTP 200 


  max_tokens=1       -> HTTP 200 


Note `max_tokens=1` is accepted here. The Responses API enforces a minimum of
16 — another reason the two surfaces are not interchangeable.

## 4. Streaming

Chat Completions streams `data: {...}` SSE frames carrying
`choices[0].delta.content`, terminated by `data: [DONE]`.

In [5]:
stream = client.chat.completions.create(
    model=VISION,
    messages=[{"role": "user", "content": "List four business uses for document image understanding."}],
    max_tokens=300,
    stream=True,
)
chunks = 0
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        chunks += 1
        print(delta, end="", flush=True)
print(f"\n\n[{chunks} content deltas received]")

 Here are four business uses for document image understanding:

1. Streamlined Foraging Operations:

Automated document analysis can significantly speed up the foraging process by quickly recognizing important information in

 planning documents, headers, and distinction between main articles and appendices. This allows for faster decision-making and more efficient resource allocation, ultimately increasing productivity.

2. Optimal crop

 yield

Document image understanding can aid incase studies and italicized conversations about the potential success,复发率, and new intuitions after losing shirts in market this year.

 By analyzing past documentation, businesses can make informed antioxidizing claims to maintain a good reputation.



[4 content deltas received]


## 5. Multi-turn — you manage the history

No server-side state on this API. Append each turn yourself.

In [6]:
# NOTE: this model requires strictly alternating user/assistant roles
# and rejects a leading `system` message with a 400. Put any instruction
# into the user turn instead.
messages = [
    {"role": "user", "content": "What is OCR?"},
]
first = client.chat.completions.create(model=VISION, messages=messages, max_tokens=200)
print("assistant:", first.choices[0].message.content)

messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append({"role": "user", "content": "How does a vision-language model differ from plain OCR?"})

second = client.chat.completions.create(model=VISION, messages=messages, max_tokens=200)
print("\nassistant:", second.choices[0].message.content)
print(f"\ninput tokens grew: {first.usage.prompt_tokens} -> {second.usage.prompt_tokens}")

assistant:  OCR stands for Optical Character Recognition. It's a technology that allows computers to read text from images. OCR can be used to convert various types of documents, including books,报纸,电子邮件 and website content, into digital formats that can be searched or processed electronically. While it's images or audio that are typically processed, the term " optical " in the name refers to the human visual system's ability to interpret and interpret visual information. However, neither the human eye nor the human body is involved in the underlying process of Optical Character Recognition.



assistant:  Vision-language models differ significantly from plain OCR in several key ways:

1. Contextual understanding: Vision-language models combine visual information with language processing, allowing them to understand the context and intent of the text they're analyzing. They can analyze how text relates to the surrounding image elements.

2. Visual reasoning: These models are capable of performing tasks that require visual reasoning. They can understand位置 relation and quietly answer questions based on the visual information in the image.

3.ality cross-modal alignment: Vision-language models Music not only can understand text in images but also align text with the visual content.

4. Large-scale pre-training: They typically require extensive pre-training on large datasets, often combining image and text data together.

5. Applicability: While OCR is limited to text-based documents, vision-language models can analyze and process a wider variety of visual content, including com

That growth is the cost of client-side history. Families on the Responses API can
avoid it with `previous_response_id` (see `../03-google-gemma/`), at the price of
30-day server-side retention.

## 6. Reasoning effort

`reasoning_effort` is accepted. The trace is not returned — but the token count
moves, which is how you can tell the model really is thinking harder.

In [7]:
print(f"{'effort':10} {'status':>7} {'completion tokens':>18}")
print("-" * 38)
for effort in ("none", "low", "medium", "high"):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {"model": VISION,
         "messages": [{"role": "user", "content": "If a chart shows sales of 10, 20 and 60 for three months, what is the growth rate month over month? Show your working."}],
         "max_tokens": 400, "reasoning_effort": effort},
        region=REGION,
    )
    tokens = (data.get("usage") or {}).get("completion_tokens", "-")
    print(f"  {effort:8} {code:>7} {tokens!s:>18}")

effort      status  completion tokens
--------------------------------------


  none         200                303


  low          200                243


  medium       200                201


  high         200                102


In [8]:
# THE LIMITATION, demonstrated rather than asserted: Palmyra Vision
# rejects tool definitions. Its model card lists "Client-side tool
# calling: Not Supported", and the API agrees.
probe_tool = [{"type": "function", "function": {
    "name": "noop", "description": "does nothing",
    "parameters": {"type": "object",
                   "properties": {"x": {"type": "string"}},
                   "required": ["x"]}}}]

code, data = post(
    f"{PREFIX}/chat/completions",
    {"model": VISION, "messages": [{"role": "user", "content": "Call noop."}],
     "max_tokens": 64, "tools": probe_tool},
    region=REGION,
)
print(f"tools on palmyra-vision -> HTTP {code}")
print("message:", err(data)[:150])
print("\n=> No forced-tool trick here. Use response_format instead (see below).")

tools on palmyra-vision -> HTTP 400
message: ErrorEvent { error: APIError { type: "BadRequestError", code: Some(400), message: "\"auto\" tool choice requires --enable-auto-tool-choice and --tool-

=> No forced-tool trick here. Use response_format instead (see below).


## 7. Tool use — not available on this model

This family **rejects tool definitions**. Its model card lists "Client-side tool
calling: Not Supported", and the API returns a 400. We show it rather than
leaving you to discover it.

The consequence: the usual "forced tool call for strict JSON" trick is
unavailable here, so `response_format` (next section) is your only structured
output route.

## 8. Structured output with `response_format`

Two variants: loose `json_object`, and schema-enforced `json_schema`.

### Budget enough tokens, or you get nothing

A reasoning-capable model may spend most of its budget thinking before it emits
the opening brace. If `max_tokens` runs out first you get **HTTP 200 with empty
content** and `finish_reason="length"` - not an error, just nothing usable.
Always check `finish_reason` before parsing.

In [9]:
def json_object_call(prompt, max_tokens, model=VISION):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {"model": model, "messages": [{"role": "user", "content": prompt}],
         "max_tokens": max_tokens, "response_format": {"type": "json_object"}},
        region=REGION,
    )
    choice = (data.get("choices") or [{}])[0]
    content = choice.get("message", {}).get("content") or ""
    return code, choice.get("finish_reason"), content


PROMPT = "Give the capital and population of France as JSON."
for budget in (64, 600):
    code, finish, content = json_object_call(PROMPT, budget)
    print(f"max_tokens={budget:4} HTTP {code} finish={finish!s:8} "
          f"content_len={len(content)}")
    if finish == "length" and not content.strip():
        print("    -> truncated before any JSON was emitted; raise max_tokens")
    elif content.strip():
        print("    ->", parse_json_lenient(content))

max_tokens=  64 HTTP 200 finish=stop     content_len=50
    -> {'capital': 'Paris', 'population': 67254859}


max_tokens= 600 HTTP 200 finish=stop     content_len=50
    -> {'Capital': 'Paris', 'Population': 67010000}


In [10]:
schema = {
    "type": "object",
    "properties": {
        "country": {"type": "string"},
        "capital": {"type": "string"},
        "population_millions": {"type": "number"},
    },
    "required": ["country", "capital", "population_millions"],
    "additionalProperties": False,
}

code, data = post(
    f"{PREFIX}/chat/completions",
    {"model": VISION,
     "messages": [{"role": "user", "content": "Describe France."}],
     "max_tokens": 250,
     "response_format": {"type": "json_schema",
                          "json_schema": {"name": "country", "strict": True,
                                           "schema": schema}}},
    region=REGION,
)
choice = (data.get("choices") or [{}])[0]
content = choice.get("message", {}).get("content")   # may be None!
print("json_schema ->", code, "| finish_reason:", choice.get("finish_reason"))
print("raw:", repr((content or "")[:160]))

if choice.get("finish_reason") == "length":
    # Reasoning consumed the budget before the object closed. Retry bigger.
    print("truncated - retrying with a larger budget")
    code, data = post(
        f"{PREFIX}/chat/completions",
        {"model": VISION,
         "messages": [{"role": "user", "content": "Describe France."}],
         "max_tokens": 2000,
         "response_format": {"type": "json_schema",
                              "json_schema": {"name": "country", "strict": True,
                                               "schema": schema}}},
        region=REGION,
    )
    choice = (data.get("choices") or [{}])[0]
    content = choice.get("message", {}).get("content")
    print("retry finish_reason:", choice.get("finish_reason"))

parsed = parse_json_lenient(content or "")
print("parsed:", json.dumps(parsed, indent=2))
assert set(parsed) >= {"country", "capital"}, parsed

json_schema -> 200 | finish_reason: stop
raw: '{\n  "country": "France",\n  "capital": "Paris",\n  "population_millions": 67.874337980567286567014247785507672495903498325421447062062507421\n  }'
parsed: {
  "country": "France",
  "capital": "Paris",
  "population_millions": 67.87433798056729
}


**Always parse leniently.** Even in strict mode, some mantle models append
characters after a valid object (Gemma 4 does this in ~half of runs), which makes
a bare `json.loads()` raise on output that is otherwise fine.

## 9. Compare the models in this family

Only one model in this family, so we compare prompting strategies instead of models.

In [11]:
task = "In one sentence, when is a specialised vision model preferable to a general multimodal one?"

print(f"{'model':44} {'latency':>9} {'out tok':>8}  answer")
print("-" * 108)
for model in [VISION]:
    started = time.perf_counter()
    code, data = post(
        f"{PREFIX}/chat/completions",
        {"model": model, "messages": [{"role": "user", "content": task}],
         "max_tokens": 160},
        region=REGION,
    )
    elapsed = time.perf_counter() - started
    if code != 200:
        print(f"{model:44} {'-':>9} {'-':>8}  HTTP {code}: {err(data)[:40]}")
        continue
    text = (data["choices"][0]["message"]["content"] or "").strip().replace("\n", " ")
    print(f"{model:44} {elapsed:>8.2f}s "
          f"{data['usage']['completion_tokens']:>8}  {text[:44]!r}")

model                                          latency  out tok  answer
------------------------------------------------------------------------------------------------------------


writer.palmyra-vision-7b                         1.02s       40  'A specialised vision model is preferable to '


## 10. Latency: TTFT and throughput

TTFT is dominated by *prefill* (the model reading your prompt) plus queue time.
Service tiers trade cost against queue priority — they mostly separate under
contention, so single samples on an idle account look flat.
(`../00-foundations/03` has the full treatment.)

In [12]:
print(f"{'tier':10} {'TTFT (s)':>10} {'total (s)':>10} {'frames/s':>10}")
print("-" * 44)
for tier in ("default", "flex", "priority"):
    m = ttft(
        f"{PREFIX}/chat/completions",
        {"model": VISION,
         "messages": [{"role": "user", "content": "List four business uses for document image understanding."}],
         "max_tokens": 200, "service_tier": tier},
        region=REGION,
    )
    if m.get("error"):
        print(f"{tier:10} {m['error']:>32}  (tier not supported by this model)")
    else:
        print(f"{tier:10} {m['ttft_s']:>10.3f} {m['total_s']:>10.3f} "
              f"{m['frames_per_s']:>10.1f}")

tier         TTFT (s)  total (s)   frames/s
--------------------------------------------


default         1.001      1.078       38.9


flex            1.181      1.194      149.4


priority        0.980      1.944        7.3


## 11. Production hardening

Retries, cost attribution, and privacy. Mantle has **no RPM quota** — throttling
is token-based, and most models here have no published TPM quota at all, so
capacity is fair-share. That makes retry-with-backoff mandatory, not optional.

In [13]:
code, project = post(
    "/v1/organization/projects",
    {"name": "palmyra-samples",
     "tags": {"Application": "PalmyraDemo", "Environment": "Demo"}},
    region=REGION,
)
project_id = project.get("id")
print("project:", code, project_id)

code, data = post(
    f"{PREFIX}/chat/completions",
    {"model": VISION,
     "messages": [{"role": "user", "content": "Reply OK"}], "max_tokens": 16},
    region=REGION,
    headers={"OpenAI-Project": project_id},   # cost attribution
)
print("attributed call ->", code)

project: 200 proj_3rejjpehoox3t6jy32lv


attributed call -> 200


In [14]:
class PalmyraClient:
    """Production-shaped wrapper for this family on bedrock-mantle."""

    def __init__(self, model=VISION, region=REGION, tier="default", project=None):
        self.model, self.region, self.tier, self.project = model, region, tier, project

    def chat(self, messages, *, max_tokens=512, schema=None, effort=None):
        # NOTE: no `tools` param (rejected) and no `system` role (rejected).
        body = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": 0.7,
            "service_tier": self.tier,
        }
        if effort:
            body["reasoning_effort"] = effort
        if schema:
            body["response_format"] = {
                "type": "json_schema",
                "json_schema": {"name": "out", "strict": True, "schema": schema},
            }
        headers = {"OpenAI-Project": self.project} if self.project else None
        # post() retries 429 + 5xx with exponential backoff and jitter.
        code, data = post(f"{PREFIX}/chat/completions", body,
                          region=self.region, headers=headers)
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        return data

    def json(self, prompt, schema, **kw):
        data = self.chat([{"role": "user", "content": prompt}], schema=schema, **kw)
        return parse_json_lenient(data["choices"][0]["message"]["content"])


bot = PalmyraClient(tier="flex", project=project_id)
out = bot.json(
    "Name the largest ocean and its average depth in metres.",
    {"type": "object",
      "properties": {"ocean": {"type": "string"}, "avg_depth_m": {"type": "number"}},
      "required": ["ocean", "avg_depth_m"], "additionalProperties": False},
)
print("structured result:", out)

structured result: {'ocean': 'Pacific Ocean', 'avg_depth_m': 4000}


In [15]:
code, archived = post(f"/v1/organization/projects/{project_id}/archive", {}, region=REGION)
print("archived demo project:", code, archived.get("status"))

archived demo project: 200 archived


## Gotchas — Writer Palmyra Vision on bedrock-mantle

| Gotcha | Detail |
|---|---|
| Path prefix | Bare `/v1`, **not** `/openai/v1` (that's gemma-4 / gpt-5.x / grok) |
| Responses API | Returns **400** for this family — Chat Completions only |
| History | No `previous_response_id`; you send `messages` every turn |
| Reasoning trace | `reasoning_effort` works but the trace is never returned |
| Strict JSON | Parse leniently — models can append text after a valid object |
| Sampling | `temperature` **and** `top_p` both fine here; not true family-wide |
| `max_tokens` | 1 is valid here; Responses API demands ≥16 |
| `content` can be `None` | Check `finish_reason` before slicing/parsing content |
| Quotas | No RPM quota; most models have no published TPM — retry with backoff |
| `reserved` tier | Rejected as a parameter; arranged via your account team |
| CloudWatch | Metrics land in `AWS/BedrockMantle`, not `AWS/Bedrock` |
| **Tool calling** | **Rejected with 400** — matches the model card |
| Structured output | `response_format` works; use it instead of forced tools |
| **`system` role** | **Rejected** — roles must strictly alternate user/assistant |

## Where next
- Same API shape: `../04-qwen/` (qwen3-vl), `../10-nvidia-nemotron/` (nano-12b-v2)
- Different API shape: `../03-google-gemma/` (Responses),
  `../02-anthropic-claude/` (Messages), `../01-openai-gpt/` (web search, caching)
- Shared mechanics: `../00-foundations/`